# 02 — Training and ablation (Colab GPU)

Trains the experiment chain **EXP-001 → EXP-008**: the YOLO baseline, then one component added per run.

Each run writes a row to `results/experiments.jsonl` recording its config, seed, environment and metrics — that ledger is the only source the paper's tables read from.

**Start with `EPOCHS = 2` to prove the whole chain runs, then raise it.** A run that fails at epoch 90 costs everything; a run that fails at epoch 2 costs nothing.

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'Enable a GPU runtime first.'
WORK = '/content/sarr-imaging'
if not os.path.isdir(WORK):
    !git clone --depth 1 https://github.com/officialarghya29/sarr-imaging.git $WORK
%cd $WORK
!pip -q install ultralytics pandas
print('GPU:', torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, 'GB VRAM')

## 1. Configure

Generate the model YAMLs and the experiment configs for the dataset you prepared. Keep the batch size fixed across runs so the comparison is controlled; lower it if you hit OOM, and lower `IMGSZ` only if necessary (it changes the small-object regime).

In [ ]:
DATASET = 'ssdd'    # must already be prepared by notebook 01
SCALE   = 's'       # n | s | m | l
EPOCHS  = 2         # raise to 100 for real results
IMGSZ   = 640
BATCH   = 16
SEED    = 0

%cd /content/sarr-imaging
!python -m saryolo arch --variant all --nc 1 --out configs/models
!python scripts/make_exp_configs.py --dataset $DATASET --scale $SCALE --epochs $EPOCHS --imgsz $IMGSZ --batch $BATCH
!ls configs/exp

## 2. Sanity-check the architecture before spending GPU time

Confirms the two invariants the whole ablation depends on: the baseline is *exactly* stock YOLO11, and every SAR module is an exact identity at initialisation.

In [ ]:
!python -m pytest tests/ -q

In [ ]:
# Params/FLOPs for the architecture table -- needs no training at all.
VARS = ' '.join(f'yolo11{SCALE}_{v}' for v in ('baseline', 'sfe', 'speckle', 'attention', 'amf', 'p2', 'full'))
!python -m saryolo bench --variants {VARS} --nc 1

## 3. Train the chain

EXP-001…EXP-008. The matrix skips EXP-009/010/011 because those are *evaluation* studies that reuse the EXP-007 checkpoint — training them separately would waste GPU hours.

`--keep-going` continues past a failure so one bad config does not cost the whole session.

In [ ]:
!python scripts/train_all_experiments.py --keep-going --only EXP-001 EXP-002 EXP-003 EXP-004

In [ ]:
!python scripts/train_all_experiments.py --keep-going --only EXP-005 EXP-006 EXP-007 EXP-008

## 4. Inspect the ledger

`TBD` means the run did not complete and produced no metric — it is never a zero. Only `completed` rows are eligible for a paper table.

In [ ]:
!python -m saryolo ledger

In [ ]:
import json, pandas as pd
rows = [json.loads(line) for line in open('results/experiments.jsonl')]
key = ['experiment_id', 'model', 'status', 'train_seed', 'epochs', 'batch', 'imgsz']
metrics = ['mAP50', 'mAP50_95', 'precision', 'recall', 'params_M', 'flops_G', 'fps']
df = pd.DataFrame([{**{k: r.get(k) for k in key}, **{m: r.get('metrics', {}).get(m) for m in metrics}} for r in rows])
df = df[df.status == 'completed'].drop_duplicates('experiment_id', keep='last')
display(df.sort_values('experiment_id'))

## 5. Scale-wise evaluation (the paper's core measurement)

`mAP` alone can hide exactly the small-object effect this project claims, so `AP_small`/`AP_medium`/`AP_large` are measured separately. A range with no ground truth returns `null` — not measurable, not zero.

In [ ]:
import glob, json
for exp, tag in (('EXP-001', 'baseline'), ('EXP-007', 'saryolo')):
    weights = glob.glob(f'results/runs/{exp}/weights/best.pt')
    if not weights:
        print(f'{exp}: no checkpoint (run not completed)'); continue
    !python -m saryolo eval --weights {weights[0]} --data configs/datasets/{DATASET}.yaml --imgsz {IMGSZ} --out results/eval/{exp}_{tag}
    print(exp, json.load(open(f'results/eval/{exp}_{tag}/metrics.json')))

## 6. Generate the ablation tables

Tables are built from the ledger only, and unmeasured cells render as `TBD`. `--require-complete` is the submission gate: it fails rather than letting an incomplete table look finished.

In [ ]:
!python -m saryolo assets
from IPython.display import Markdown, display
display(Markdown(open('paper/tables/main_ablation.md').read()))
display(Markdown(open('paper/tables/scale_analysis.md').read()))

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/saryolo/results && cp -r results /content/drive/MyDrive/saryolo/
    print('saved results to Drive')
print('\nNext: notebooks/03_benchmark_and_paper.ipynb')